In [ ]:
import os
from pathlib import Path
from src.utils import *
from src.config import PROCESSED_IMG_DIR, RAW_IMG_DIR
from src.feature_extraction import *
from src.feature_extraction import _create_fixed_box
from src.region_visualization import collect_all_rois, build_mosaic, show_mosaic
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import f_classif
from mpl_toolkits.mplot3d import Axes3D
import pandas as pd
from src.config import ROOT_DIR
import time
import cv2
import random
from multiprocessing import Pool, cpu_count
from functools import partial
from src.feature_extraction import _extract_features_for_image

In [ ]:
paths = get_all_image_paths(directory = PROCESSED_IMG_DIR)
separated_paths = seprate_processed_files(paths) # Separate all paths that end in .png
processed_paths = separated_paths[0] # These are the processed images (paths only)
masks_paths = separated_paths[1]     # These are the binary masks (paths only)
skeletons_paths = separated_paths[2] # These are the skeletons (paths only)
xml_paths = []                       # And this for the .xml (paths only)
ground_truth_bboxes = []             # And this for the ground truth bboxes
for img_path in processed_paths:
    xml_path = get_xml_path(img_path)
    xml_path = xml_path.replace('_processed.xml', '_bbox.xml') # Fix the ending from _processed.xml to _bbox.xml
    xml_paths.append(xml_path)
    bbox = parse_stenosis_xml(xml_path)
    ground_truth_bboxes.append(bbox)
print('XML files: ', len(xml_paths))
print('Ground truth bounding boxes: ', len(ground_truth_bboxes))

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 0b: Build the 1498-image subset (7 frames × 214 series)
# Strategy: for each serie of each patient, select the 7 frames
# centred around the middle frame of that serie.
# ─────────────────────────────────────────────────────────────
from collections import defaultdict


FRAMES_PER_SERIE = 7  # min-of-maxs determined from dataset analysis
STAT_NAMES  = ['mean', 'var', 'entropy', 'energy', 'kurtosis', 'skewness']
TILE_LABELS = ['tl', 'tr', 'bl', 'br']
ROI_SIZE    = 80
HALF_SIZE   = ROI_SIZE // 2
LABELING_THRESHOLD = 0.50
N_SIZES = 6
N_ORIENTATIONS = 12

# ── Step 1: Group processed paths by patient and serie ───────
# Path structure: .../dataset_processed/<patientID>/<serieID>/<frame>_processed.png
patient_series = defaultdict(lambda: defaultdict(list))

# After initialising patient_series, add:
for path in processed_paths:
    p          = Path(path)
    patient_id = p.parts[-3]
    serie_id   = p.parts[-2]
    patient_series[patient_id][serie_id].append(path)

# Sort frames within each serie
for patient_id in patient_series:
    for serie_id in patient_series[patient_id]:
        patient_series[patient_id][serie_id] = sorted(patient_series[patient_id][serie_id])

# ── Step 2: Select 7 centre frames from every serie ──────────
subset_processed_paths_b = []

for patient_id, series in sorted(patient_series.items()):
    for serie_id, frame_paths in sorted(series.items()):
        n    = len(frame_paths)
        mid  = n // 2
        half = FRAMES_PER_SERIE // 2

        start = mid - half
        end   = start + FRAMES_PER_SERIE

        # Clamp to valid range
        start = max(0, start)
        end   = min(n, start + FRAMES_PER_SERIE)
        start = max(0, end - FRAMES_PER_SERIE)  # re-adjust if end was clamped

        selected = frame_paths[start:end]
        subset_processed_paths_b.extend(selected)

print(f"Total series found    : {sum(len(s) for s in patient_series.values())}")
print(f"Total frames selected : {len(subset_processed_paths_b)}  (expected 7 × 214 = 1498)")

# ── Step 3: Derive aligned mask, skeleton and GT bbox lists ──
subset_masks_paths_b         = [get_skeleton_path(p).replace('_skeleton.png', '_mask.png') for p in subset_processed_paths_b]
subset_skeletons_paths_b     = [get_skeleton_path(p) for p in subset_processed_paths_b]
subset_ground_truth_bboxes_b = [parse_stenosis_xml(get_bbox_xml_path(p)) for p in subset_processed_paths_b]

print(f"Masks paths            : {len(subset_masks_paths_b)}")
print(f"Skeleton paths         : {len(subset_skeletons_paths_b)}")
print(f"GT bbox lists          : {len(subset_ground_truth_bboxes_b)}")
print(f"Images with annotations: {sum(1 for b in subset_ground_truth_bboxes_b if len(b) > 0)}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL A: Extract ROI coordinates → roi_coordinates_subset_b.csv
# Uses the 1498-image subset built in Cell 0b.
# ─────────────────────────────────────────────────────────────
OUTPUT_DIR_CSV_SUBSET_B = Path(os.path.join(ROOT_DIR, "notebooks\csv_files\subset_1498images"))
OUTPUT_DIR_CSV_SUBSET_B.mkdir(parents=True, exist_ok=True)

subset_dataset_b = list(zip(subset_processed_paths_b, subset_masks_paths_b,
                            subset_skeletons_paths_b, subset_ground_truth_bboxes_b))

rows_coords_subset_b = []

print(f"Starting coordinate extraction over {len(subset_dataset_b)} images...")
start = time.time()

for img_path, mask_path, skel_path, gt_boxes in subset_dataset_b:

    img      = load_image(img_path)
    skeleton = load_image(skel_path)
    if img is None or skeleton is None:
        continue

    img_shape = img.shape[:2]

    p          = Path(img_path)
    patient_id = p.parts[-3]
    serie_id   = p.parts[-2]
    frame_stem = p.stem.replace('_processed', '')
    image_name = f"{patient_id}_{serie_id}_{frame_stem}"

    sampled_boxes  = extract_rois(skeleton, img_shape, roi_size=ROI_SIZE, total_rois=100)
    sampled_labels = label_rois(sampled_boxes, gt_boxes, threshold=LABELING_THRESHOLD)

    all_boxes_and_labels = list(zip(sampled_boxes, sampled_labels))

    for gt_box in gt_boxes:
        cx    = (gt_box['xmin'] + gt_box['xmax']) // 2
        cy    = (gt_box['ymin'] + gt_box['ymax']) // 2
        fixed = _create_fixed_box(cx, cy, HALF_SIZE, img_shape)
        all_boxes_and_labels.append((fixed, 1))

    for roi_idx, (box, lbl) in enumerate(all_boxes_and_labels, start=1):
        rows_coords_subset_b.append({
            'roi_name'   : f"{image_name}_{roi_idx}",
            'image_name' : image_name,
            'xmin'       : box['xmin'],
            'ymin'       : box['ymin'],
            'xmax'       : box['xmax'],
            'ymax'       : box['ymax'],
            'label'      : lbl,
        })

elapsed = time.time() - start
df_coords_subset_b = pd.DataFrame(rows_coords_subset_b)

print(f"\nDone in {elapsed:.1f}s — {len(rows_coords_subset_b)} ROIs extracted.")
print(f"df_coords_subset_b shape : {df_coords_subset_b.shape}")
print(f"Stenosis ROIs            : {df_coords_subset_b['label'].sum()}")
print(f"Healthy ROIs             : {(df_coords_subset_b['label'] == 0).sum()}")

coords_csv_path_subset_b = OUTPUT_DIR_CSV_SUBSET_B / "roi_coordinates_subset_b.csv"
df_coords_subset_b.to_csv(coords_csv_path_subset_b, index=False)
print(f"\nSaved: {coords_csv_path_subset_b}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL B: Extract features for each ROI → roi_features_subset_b.csv
# Parallelised across all available CPU cores.
# Requires df_coords_subset_b from Cell A to be in memory.
# ─────────────────────────────────────────────────────────────
subset_dataset_b = list(zip(subset_processed_paths_b, subset_masks_paths_b,
                            subset_skeletons_paths_b, subset_ground_truth_bboxes_b))

gabor_bank = build_gabor_bank(ksize=31, n_sizes=N_SIZES, n_orientations=N_ORIENTATIONS)
feature_cols = build_feature_column_names(
    n_filters=len(gabor_bank),
    stat_names=STAT_NAMES,
    tile_labels=TILE_LABELS,
)

worker_fn_subset_b = partial(
    _extract_features_for_image,
    roi_size           = ROI_SIZE,
    half_size          = HALF_SIZE,
    labeling_threshold = LABELING_THRESHOLD,
    gabor_bank         = gabor_bank,
    feature_cols       = feature_cols,
)

n_cores = cpu_count()
print(f"Starting parallel feature extraction over {len(subset_dataset_b)} images using {n_cores} cores...")
start = time.time()

with Pool(processes=n_cores) as pool:
    results = pool.map(worker_fn_subset_b, subset_dataset_b)

rows_features_subset_b = [row for image_rows in results for row in image_rows]

elapsed = time.time() - start
df_features_subset_b = pd.DataFrame(rows_features_subset_b)

print(f"\nDone in {elapsed:.1f}s — {len(rows_features_subset_b)} ROIs extracted.")
print(f"df_features_subset_b shape : {df_features_subset_b.shape}")
print(f"Stenosis ROIs              : {df_features_subset_b['label'].sum()}")
print(f"Healthy ROIs               : {(df_features_subset_b['label'] == 0).sum()}")

features_csv_path_subset_b = OUTPUT_DIR_CSV_SUBSET_B / "roi_features_subset_b.csv"
df_features_subset_b.to_csv(features_csv_path_subset_b, index=False)
print(f"\nSaved: {features_csv_path_subset_b}")

## EXPLORATION OF EXTRACTED FEATURES

In [ ]:
# ─────────────────────────────────────────────────────────────
# EXPLORATION: Load CSVs and reconstruct feature matrix
# ─────────────────────────────────────────────────────────────
df_features_1498 = pd.read_csv(features_csv_path_subset_b)

y_1498        = df_features_1498['label'].values
X_1498        = df_features_1498.drop(columns=['roi_name', 'label']).values
feature_cols_1498 = [c for c in df_features_1498.columns if c not in ('roi_name', 'label')]

scaler_1498   = StandardScaler()
X_scaled_1498 = scaler_1498.fit_transform(X_1498)

print(f"Loaded feature matrix : {X_1498.shape}")
print(f"Stenosis ROIs         : {np.sum(y_1498)}")
print(f"Healthy ROIs          : {len(y_1498) - np.sum(y_1498)}")
print(f"Class imbalance       : {round((len(y_1498) - np.sum(y_1498)) / np.sum(y_1498), 2)}× more healthy than stenosis")

In [ ]:
# ─────────────────────────────────────────────────────────────
# EXPLORATION: 3D PCA scatter plot
# ─────────────────────────────────────────────────────────────
%matplotlib tk

pca_3d_1498   = PCA(n_components=3)
X_pca_3d_1498 = pca_3d_1498.fit_transform(X_scaled_1498)
total_variance_1498 = pca_3d_1498.explained_variance_ratio_.sum() * 100

fig = plt.figure(figsize=(10, 8))
ax  = fig.add_subplot(111, projection='3d')

ax.scatter(X_pca_3d_1498[y_1498 == 0, 0], X_pca_3d_1498[y_1498 == 0, 1], X_pca_3d_1498[y_1498 == 0, 2],
           alpha=0.4, label='Healthy (0)',  color='#3498db', s=25)
ax.scatter(X_pca_3d_1498[y_1498 == 1, 0], X_pca_3d_1498[y_1498 == 1, 1], X_pca_3d_1498[y_1498 == 1, 2],
           alpha=0.8, label='Stenosis (1)', color='#e74c3c', marker='x', s=45)

ax.set_title(f"3D PCA — 1498-Image Subset\n(Total Explained Variance: {total_variance_1498:.1f}%)",
             fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel(f"PC 1 ({pca_3d_1498.explained_variance_ratio_[0]*100:.1f}%)", fontsize=10, labelpad=10)
ax.set_ylabel(f"PC 2 ({pca_3d_1498.explained_variance_ratio_[1]*100:.1f}%)", fontsize=10, labelpad=10)
ax.set_zlabel(f"PC 3 ({pca_3d_1498.explained_variance_ratio_[2]*100:.1f}%)", fontsize=10, labelpad=10)
ax.xaxis.set_pane_color((0.95, 0.95, 0.95, 1.0))
ax.yaxis.set_pane_color((0.95, 0.95, 0.95, 1.0))
ax.zaxis.set_pane_color((0.95, 0.95, 0.95, 1.0))
ax.view_init(elev=25, azim=45)
ax.legend(loc='upper left', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# EXPLORATION: Feature ranking and boxplots
# ─────────────────────────────────────────────────────────────
%matplotlib inline

f_values_1498, p_values_1498 = f_classif(X_scaled_1498, y_1498)

n_indeces = 8
top_n_indices_1498 = np.argsort(f_values_1498)[::-1][:n_indeces]

print(f"\n--- TOP {n_indeces} MOST DISCRIMINATIVE FEATURES — 1498-Image Subset ---")
for rank, idx in enumerate(top_n_indices_1498, start=1):
    print(f"Rank {rank}: Feature {idx:4d}  '{feature_cols_1498[idx]}'  — F-score: {f_values_1498[idx]:.2f}  p: {p_values_1498[idx]:.2e}")

labels_1498 = np.where(y_1498 == 0, 'Healthy', 'Stenosis')

fig, axes = plt.subplots(1, n_indeces, figsize=(16, 5))
fig.suptitle(f"Top {n_indeces} Most Discriminative Features — 1498-Image Subset", fontsize=16)

for i, feat_idx in enumerate(top_n_indices_1498):
    ax = axes[i]
    sns.boxplot(
        x=labels_1498,
        y=X_scaled_1498[:, feat_idx],
        hue=labels_1498,
        order=['Healthy', 'Stenosis'],
        hue_order=['Healthy', 'Stenosis'],
        palette={'Healthy': '#3498db', 'Stenosis': '#e74c3c'},
        legend=False,
        ax=ax,
    )
    ax.set_title(feature_cols_1498[feat_idx], fontsize=7, wrap=True)
    ax.set_xlabel("")
    ax.set_ylabel("Standardised Value" if i == 0 else "")

plt.tight_layout()
plt.show()